# Week 02 · Day 01 — LangChain Fundamentals

**Topics:** LCEL pipelines · ChatPromptTemplate · Memory · Async & streaming


In [ ]:
%pip install -q langchain langchain-openai langchain-community

In [ ]:
import os
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2, api_key=os.getenv("OPENAI_API_KEY"))

## 1 · LCEL — The Pipe Operator

LangChain Expression Language (LCEL) chains components together using the `|` pipe operator. Each component receives the output of the previous one.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Build a chain: prompt | llm | parser
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {domain}. Be concise."),
    ("human", "{question}"),
])

chain = prompt | llm | StrOutputParser()

result = chain.invoke({
    "domain": "Python programming",
    "question": "What is a generator and when should I use one?",
})
print(result)

In [ ]:
# Batch invocation — runs multiple inputs efficiently
questions = [
    {"domain": "Python", "question": "What is a decorator?"},
    {"domain": "databases", "question": "What is an index?"},
    {"domain": "networking", "question": "What is a TCP handshake?"},
]

results = chain.batch(questions)
for q, r in zip(questions, results):
    print(f"Q ({q['domain']}): {q['question']}")
    print(f"A: {r[:100]}\n")

## 2 · Output Parsers

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field

class MovieReview(BaseModel):
    title: str = Field(description="Movie title")
    rating: float = Field(description="Rating from 0.0 to 10.0")
    summary: str = Field(description="One sentence summary")
    recommended: bool = Field(description="Whether to recommend")

parser = JsonOutputParser(pydantic_object=MovieReview)

review_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract movie review information. {format_instructions}"),
    ("human", "{review_text}"),
]).partial(format_instructions=parser.get_format_instructions())

review_chain = review_prompt | llm | parser

review_text = "Inception (2010) by Christopher Nolan is a mind-bending thriller that deserves a 9/10. The layered dream sequences are masterfully executed. Highly recommended for sci-fi fans."

result = review_chain.invoke({"review_text": review_text})
print(result)

## 3 · Conversation Memory

Use `RunnableWithMessageHistory` to persist chat history across turns.

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# In-memory store — use Redis/DynamoDB for production
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful coding assistant."),
    ("placeholder", "{history}"),
    ("human", "{input}"),
])

chat_chain = chat_prompt | llm | StrOutputParser()

with_history = RunnableWithMessageHistory(
    chat_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

config = {"configurable": {"session_id": "user-123"}}

# Multi-turn conversation
turns = [
    "My name is Alice and I'm learning Python.",
    "What's a good first project for me?",
    "Can you remind me what my name is?",  # tests memory
]

for turn in turns:
    response = with_history.invoke({"input": turn}, config=config)
    print(f"Human: {turn}")
    print(f"AI:    {response[:120]}\n")

## 4 · Async and Streaming

In [ ]:
import asyncio

# Async invoke — releases the event loop while waiting
async def async_demo():
    result = await chain.ainvoke({
        "domain": "machine learning",
        "question": "What is overfitting?",
    })
    return result

result = asyncio.run(async_demo())
print(result)

In [ ]:
# Async parallel — run multiple chains concurrently
async def parallel_demo():
    tasks = [
        chain.ainvoke({"domain": "Python", "question": "What is a context manager?"}),
        chain.ainvoke({"domain": "SQL", "question": "What is a JOIN?"}),
        chain.ainvoke({"domain": "Docker", "question": "What is a container layer?"}),
    ]
    return await asyncio.gather(*tasks)

import time
t0 = time.perf_counter()
results = asyncio.run(parallel_demo())
print(f"3 parallel calls in {time.perf_counter() - t0:.1f}s")
for r in results:
    print(f"  {r[:80]}")

In [ ]:
# Streaming — yield tokens as they arrive
print("Streaming response:")
for chunk in chain.stream({
    "domain": "software engineering",
    "question": "What is the SOLID principle?",
}):
    print(chunk, end="", flush=True)
print()

## 5 · RunnablePassthrough and RunnableLambda

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# RunnablePassthrough: pass input through unchanged
# RunnableLambda: wrap any Python function as a chain step

def format_question(x: dict) -> dict:
    return {
        "domain": x["domain"].upper(),
        "question": x["question"].strip() + "?",
    }

preprocessing_chain = RunnableLambda(format_question) | chain

result = preprocessing_chain.invoke({
    "domain": "python",
    "question": "what is __init__",
})
print(result)

## Summary

- `prompt | llm | parser` — the canonical LCEL chain. Each `|` pipes output to the next.
- `.invoke()` for single inputs, `.batch()` for multiple, `.stream()` for streaming.
- `RunnableWithMessageHistory` wraps any chain to add per-session memory.
- `.ainvoke()` / `.astream()` are async; use `asyncio.gather()` for parallel calls.
- `RunnableLambda` turns any Python function into a chain step.

**Next:** [Advanced RAG](week02-day01-advanced-rag.ipynb)